# ĐỀ 1.1 – LỜI GIẢI CHI TIẾT

**Tệp đầu vào:** `wine.csv`, đặt cùng thư mục Notebook/script. Cột nhãn tên là `class`.

**Mẫu mới X:** `11.2, 0.5, 0.7, 2.5, 0.07, 12, 50, 1.008, 4.25, 0.57, 10`.

- Câu 1: Chuẩn hóa tất cả cột về `[0,10]` (cột nhãn chuỗi được mã hóa chỉ để tạo bảng chuẩn hóa; nhãn gốc giữ nguyên để phân lớp).
- Câu 2: Gaussian Naive Bayes **tự cài đặt**: tiên nghiệm, trung bình, phương sai, mật độ Gauss, hậu nghiệm.
- Câu 3: Kiểm nghiệm bằng `sklearn.naive_bayes.GaussianNB`.
- Câu 4: K-Means **tự cài đặt** (K=4), thống kê số mẫu/tâm cụm, suy ra cụm của X.

Không có bước chia train/test vì đề không yêu cầu: toàn bộ CSV làm dữ liệu để học, X là mẫu cần suy luận. Đây không phải đánh giá độ chính xác của mô hình trên tập kiểm tra.


In [ ]:
# CÀI ĐẶT THƯ VIỆN (chạy Terminal khi cần):
# python -m pip install numpy pandas scikit-learn

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.naive_bayes import GaussianNB

# ===== THAM SỐ CẦN SỬA CHO ĐỀ KHÁC =====
CSV_PATH = Path('wine.csv')  # str/Path: đường dẫn CSV; đặt cùng thư mục thì giữ nguyên
TARGET = 'class'            # str: tên cột nhãn trong dữ liệu
K = 4                       # int: số cụm K-Means theo đề
RANDOM_STATE = 42           # int: hạt giống ngẫu nhiên để chạy lại cho cùng kết quả
MAX_ITER = 300              # int: số vòng lặp tối đa của K-Means
TOL = 1e-4                  # float: dừng nếu tâm cụm dịch chuyển tối đa <= TOL
VAR_SMOOTHING = 1e-9        # float: hệ số cộng rất nhỏ vào phương sai của GaussianNB

# 11 số của mẫu X phải theo đúng thứ tự 11 cột đặc trưng trong file CSV.
X_RAW = np.array([
    11.2, 0.5, 0.7, 2.5, 0.07,
    12, 50, 1.008, 4.25, 0.57, 10
], dtype=float)

if not CSV_PATH.is_file():
    raise FileNotFoundError(
        f'Không tìm thấy {CSV_PATH.resolve()}. '
        'Hãy đặt wine.csv cùng thư mục đang chạy hoặc sửa CSV_PATH.'
    )

# sep=None: tự dò dấu phân cách , hoặc ; khi đọc file.
df = pd.read_csv(CSV_PATH, sep=None, engine='python')
df.columns = df.columns.str.strip()  # Bỏ dấu cách thừa đầu/cuối tên cột

if TARGET not in df.columns:
    raise KeyError(f'Không có cột {TARGET!r}. Các cột hiện tại: {df.columns.tolist()}')
if df.empty:
    raise ValueError('Tệp CSV không có dòng dữ liệu.')
if df.isna().any().any():
    raise ValueError('Dữ liệu có giá trị thiếu. Hãy xử lý thiếu trước khi làm bài.')

feature_names = [c for c in df.columns if c != TARGET]
feature_df = df[feature_names].apply(pd.to_numeric, errors='raise')
y = df[TARGET].to_numpy()  # Nhãn gốc, KHÔNG dùng nhãn đã chuẩn hóa để phân lớp

if len(feature_names) != len(X_RAW):
    raise ValueError(
        f'X gồm {len(X_RAW)} giá trị nhưng CSV có {len(feature_names)} đặc trưng. '
        'Cần kiểm tra thứ tự/các cột đặc trưng.'
    )

print('Kích thước dữ liệu:', df.shape)
print('Danh sách đặc trưng đúng thứ tự X:', feature_names)
print('Các lớp:', np.unique(y))
print('\n5 dòng đầu của dữ liệu gốc:')
print(df.head())


## Câu 1 – Chuẩn hóa tất cả các cột về [0,10]

Công thức Min-Max theo **từng cột**: 

\[x^*=10\frac{x-x_{\min}}{x_{\max}-x_{\min}}\]

- Dùng `MinMaxScaler(feature_range=(0,10))` với các cột đặc trưng.
- Để đáp ứng ý “tất cả các cột”, tạo thêm bảng gồm cả `class`. Nếu `class` là chuỗi thì mã hóa thành số tạm thời để chuẩn hóa, nhưng **không dùng** các số giả định này làm nhãn để huấn luyện.
- X được biến đổi bằng **cùng scaler** đã `fit` trên bộ dữ liệu; không được `fit` scaler mới trên X.
- Nếu cột có giá trị hằng, MinMaxScaler đưa cột đó về 0.


In [ ]:
# ----- Chuẩn hóa các đặc trưng để dùng cho Câu 2, 3, 4 -----
feature_scaler = MinMaxScaler(
    feature_range=(0, 10)  # tuple: (min, max) mong muốn cho mỗi cột
)
X_scaled = feature_scaler.fit_transform(feature_df)

# Biến đổi mẫu X bằng các min, max học được từ CSV.
x_row = pd.DataFrame([X_RAW], columns=feature_names)  # Giữ tên cột đúng thứ tự
x_scaled = feature_scaler.transform(x_row)[0]

# ----- Chuẩn hóa TOÀN BỘ cột cho yêu cầu Câu 1 -----
all_numeric = feature_df.copy()
if pd.api.types.is_numeric_dtype(df[TARGET]):
    all_numeric[TARGET] = pd.to_numeric(df[TARGET])
else:
    # Chỉ nhằm tạo bảng chuẩn hóa đủ mọi cột nếu class là nhãn dạng chữ.
    codes, labels = pd.factorize(df[TARGET], sort=True)
    all_numeric[TARGET] = codes
    print('Mã hóa tạm thời cột class khi xuất bảng:', dict(enumerate(labels)))

all_numeric = all_numeric[df.columns]  # Giữ thứ tự cột của tệp gốc
all_scaler = MinMaxScaler(feature_range=(0, 10))
df_scaled_all = pd.DataFrame(
    all_scaler.fit_transform(all_numeric),
    columns=all_numeric.columns,
    index=df.index
)

print('\nCÂU 1 – 5 dòng dữ liệu chuẩn hóa:')
print(df_scaled_all.head().round(4))
print('\nGiá trị nhỏ nhất từng cột:')
print(df_scaled_all.min().round(4))
print('\nGiá trị lớn nhất từng cột:')
print(df_scaled_all.max().round(4))

# Xuất bảng chuẩn hóa – nếu cột class ban đầu là nhãn, chỉ dùng bản xuất để tham khảo.
OUTPUT_CSV = Path('wine_chuan_hoa_0_10.csv')  # Path: tên tệp kết quả

df_scaled_all.to_csv(OUTPUT_CSV, index=False)
print('\nĐã xuất:', OUTPUT_CSV.resolve())
print('\nX đã chuẩn hóa:', np.round(x_scaled, 6))


## Câu 2 – Tự lập trình Gaussian Naive Bayes

Dữ liệu có 11 đặc trưng liên tục nên dùng phân phối Gauss. Với từng lớp `c`:

1. Đếm `n_c`, tính tiên nghiệm `P(c)=n_c/n`.
2. Tính trung bình `mu_cj` và phương sai `var_cj` của mỗi đặc trưng `j`.
3. Tính log mật độ Gauss của từng giá trị trong X theo lớp `c`.
4. Cộng log tiên nghiệm và các log mật độ để có log điểm số cho lớp.
5. Chuẩn hóa các điểm số để được xác suất hậu nghiệm `P(c|X)` (dùng log để tránh lỗi tràn số).
6. Chọn lớp có hậu nghiệm lớn nhất.

`var_smoothing` là cách tránh chia cho phương sai bằng 0 và giúp kết quả sát `GaussianNB` của scikit-learn.


In [ ]:
def gaussian_nb_thu_cong(X_train, y_train, x, var_smoothing=1e-9):
    """Tự tính xác suất hậu nghiệm theo Gaussian Naive Bayes.

    X_train: ndarray (n_samples, n_features), đặc trưng đã chuẩn hóa.
    y_train: ndarray (n_samples,), nhãn lớp gốc.
    x: ndarray (n_features,), mẫu cần phân lớp đã chuẩn hóa.
    var_smoothing: float >=0, hệ số làm trơn phương sai như GaussianNB.
    """
    classes, counts = np.unique(y_train, return_counts=True)
    n_samples = len(y_train)

    # Giống GaussianNB sklearn: epsilon = var_smoothing * max(phương sai toàn cục).
    epsilon = var_smoothing * np.var(X_train, axis=0).max()
    log_scores = []

    for c, n_class in zip(classes, counts):
        samples = X_train[y_train == c]   # Chỉ lấy mẫu thuộc lớp c
        prior = n_class / n_samples       # P(c)
        mu = samples.mean(axis=0)         # Trung bình mỗi đặc trưng theo lớp
        var = samples.var(axis=0) + epsilon  # Phương sai dân số (ddof=0)

        # log mật độ chuẩn cho từng đặc trưng của x.
        log_gaussian = -0.5 * (
            np.log(2 * np.pi * var) + (x - mu)**2 / var
        )

        # Độc lập có điều kiện => nhân các mật độ (cộng các log).
        log_score = np.log(prior) + log_gaussian.sum()
        log_scores.append(log_score)

        print(f'\nLớp {c}: số mẫu={n_class}, P(class)={prior:.6f}')
        print('  Trung bình từng đặc trưng:', np.round(mu, 4))
        print('  Phương sai từng đặc trưng:', np.round(var, 6))
        print('  Log điểm số (chưa chuẩn hóa):', round(float(log_score), 6))

    # Softmax ổn định số học: trừ điểm số lớn nhất trước khi exp.
    log_scores = np.asarray(log_scores, dtype=float)
    weights = np.exp(log_scores - log_scores.max())
    probabilities = weights / weights.sum()
    label = classes[np.argmax(probabilities)]
    return classes, probabilities, label

classes_manual, probs_manual, predicted_manual = gaussian_nb_thu_cong(
    X_train=X_scaled,
    y_train=y,
    x=x_scaled,
    var_smoothing=VAR_SMOOTHING  # float: cùng mức làm trơn với câu 3
)

print('\nCÂU 2 – XÁC SUẤT HẬU NGHIỆM:')
for c, p in zip(classes_manual, probs_manual):
    print(f'P(class={c} | X) = {p:.10f} = {p * 100:.6f}%')
print('=> Theo Naive Bayes thủ công, X thuộc lớp:', predicted_manual)


## Câu 3 – Kiểm nghiệm bằng hàm có sẵn

Dùng `GaussianNB(var_smoothing=...)`, `fit(X_scaled, y)` và `predict_proba(x_scaled.reshape(1,-1))`.

- `fit`: ước lượng các thông số phân phối từ dữ liệu.
- `predict_proba`: xuất xác suất mỗi lớp, theo thứ tự `model.classes_`.
- `predict`: xuất nhãn có xác suất lớn nhất.

Cả hai cách tính đều dùng **cùng** dữ liệu chuẩn hóa và `var_smoothing`, nên các xác suất phải gần như trùng nhau (sai số số thực rất nhỏ).


In [ ]:
model = GaussianNB(
    var_smoothing=VAR_SMOOTHING  # float: hệ số ổn định phương sai
)
model.fit(X_scaled, y)           # Học từ tất cả mẫu có nhãn trong CSV

probs_library = model.predict_proba(x_scaled.reshape(1, -1))[0]
predicted_library = model.predict(x_scaled.reshape(1, -1))[0]

print('CÂU 3 – GAUSSIANNB SCIKIT-LEARN:')
for c, p in zip(model.classes_, probs_library):
    print(f'P(class={c} | X) = {p:.10f} = {p * 100:.6f}%')
print('=> Lớp dự đoán:', predicted_library)

# Ghép kết quả đúng thứ tự lớp trước khi so sánh.
comparison = pd.DataFrame({
    'class': classes_manual,
    'thủ công': probs_manual,
    'sklearn': probs_library
})
comparison['sai_lech'] = abs(comparison['thủ công'] - comparison['sklearn'])
print('\nBảng đối chiếu Câu 2 và 3:')
print(comparison.to_string(index=False))
print('Sai lệch lớn nhất:', comparison['sai_lech'].max())
print('Hai lớp dự đoán giống nhau?', predicted_manual == predicted_library)


## Câu 4 – Tự cài đặt K-Means, K=4 (không dùng cột `class`)

Các bước lập trình:

1. Chọn ngẫu nhiên 4 dòng khác nhau trong `X_scaled` làm tâm ban đầu (được cố định với `RANDOM_STATE`).
2. Tính khoảng cách Euclid từ mỗi mẫu đến các tâm.
3. Gán mẫu vào tâm gần nhất.
4. Cập nhật tâm bằng trung bình của các mẫu thuộc cụm.
5. Lặp lại đến khi tâm thay đổi không đáng kể hoặc đạt `MAX_ITER`.
6. In số mẫu từng cụm, tọa độ tâm trong thang `[0,10]` và thang gốc; gán X vào cụm có tâm gần nhất.

**Lưu ý:** K-Means ngẫu nhiên theo khởi tạo; các tâm có thể khác nếu đổi `RANDOM_STATE`. Đánh số cụm 1–4 chỉ là quy ước.


In [ ]:
def kmeans_thu_cong(X_data, k=4, random_state=42, max_iter=300, tol=1e-4):
    """Cài đặt K-Means theo các bước gán cụm – cập nhật tâm.

    X_data: ndarray (n_samples, n_features), dữ liệu KHÔNG có class.
    k: int > 0, số cụm muốn tạo.
    random_state: int, kiểm soát lựa chọn tâm khởi tạo.
    max_iter: int > 0, số vòng lặp tối đa.
    tol: float >=0, ngưỡng dừng dựa trên dịch chuyển tâm lớn nhất.
    """
    n_samples = X_data.shape[0]
    if not (1 <= k <= n_samples):
        raise ValueError('Cần có 1 <= k <= số mẫu dữ liệu.')

    rng = np.random.default_rng(random_state)
    initial_indices = rng.choice(n_samples, size=k, replace=False)
    centers = X_data[initial_indices].copy()
    print('Chỉ số mẫu khởi tạo tâm:', initial_indices)

    for iteration in range(1, max_iter + 1):
        # Khoảng cách Euclid bình phương: (n mẫu, k tâm).
        dist_sq = np.sum((X_data[:, None, :] - centers[None, :, :])**2, axis=2)
        labels = np.argmin(dist_sq, axis=1)

        # Chữa trường hợp có cụm rỗng: chuyển mẫu xa tâm nhất từ cụm có >1 mẫu.
        for j in range(k):
            if not np.any(labels == j):
                counts = np.bincount(labels, minlength=k)
                candidates = np.where(counts[labels] > 1)[0]
                if candidates.size == 0:
                    raise RuntimeError('Không tìm được mẫu để xử lý cụm rỗng.')
                nearest_sq = dist_sq[np.arange(n_samples), labels]
                move = candidates[np.argmax(nearest_sq[candidates])]
                labels[move] = j

        new_centers = np.array([
            X_data[labels == j].mean(axis=0) for j in range(k)
        ])
        max_shift = np.max(np.linalg.norm(new_centers - centers, axis=1))
        centers = new_centers

        if max_shift <= tol:
            break

    # Phân cụm lần cuối theo tâm đã cập nhật.
    final_dist_sq = np.sum((X_data[:, None, :] - centers[None, :, :])**2, axis=2)
    labels = np.argmin(final_dist_sq, axis=1)
    return centers, labels, iteration

centers_scaled, cluster_labels, n_iterations = kmeans_thu_cong(
    X_data=X_scaled,        # ndarray: chỉ 11 đặc trưng, đã chuẩn hóa, KHÔNG có class
    k=K,                    # int: 4 cụm theo đề
    random_state=RANDOM_STATE,
    max_iter=MAX_ITER,
    tol=TOL
)

centers_original = feature_scaler.inverse_transform(centers_scaled)

print(f'\nCÂU 4 – K-MEANS THỦ CÔNG: {n_iterations} vòng lặp')
for j in range(K):
    count = np.count_nonzero(cluster_labels == j)
    print(f'\nCụm {j + 1}: {count} mẫu')
    print(' Tâm cụm trong [0, 10]:', np.round(centers_scaled[j], 4))
    print(' Tâm cụm theo thang gốc:', np.round(centers_original[j], 4))

# Gán X vào tâm gần nhất bằng khoảng cách Euclid đã chuẩn hóa.
distances_X = np.linalg.norm(centers_scaled - x_scaled, axis=1)
cluster_X = int(np.argmin(distances_X)) + 1  # +1 để đánh số cụm 1,2,3,4

print('\nKhoảng cách từ X đến từng tâm cụm:')
for j, distance in enumerate(distances_X, start=1):
    print(f'  Cụm {j}: {distance:.6f}')
print('=> X thuộc cụm:', cluster_X)
print('Tổng số mẫu trong các cụm:', len(cluster_labels))
